# Financial Markets SQL Datamart

Inserire il testo per spiegare il tutto, maxi riassunto generale

In [1]:
import os
import pandas as pd
import psycopg2
from dotenv import load_dotenv

load_dotenv()


def get_connection():
    return psycopg2.connect(
        host=os.getenv("PGHOST"),
        database=os.getenv("PGDATABASE"),
        user=os.getenv("PGUSER"),
        password=os.getenv("PGPASSWORD"),
        sslmode=os.getenv("PGSSLMODE")
    )


def execute_query(query):
    conn = None
    cur = None

    try:
        conn = get_connection()
        cur = conn.cursor()

        cur.execute(query)
        conn.commit()

        print("✓ Sweet! Data updated")

    except Exception as e:
        if conn:
            conn.rollback()

        print("Errore:", e)

    finally:
        if cur:
            cur.close()

        if conn:
            conn.close()


In [2]:
# CREAZIONE DEGLI SCHEMAS DEL PROGGETTO

# si provede con la creazione degli scheme per:

True

#Creazione degli schemas del progetto

creazione degli schemas per:
* raw
* staging
* ref
* core
* mart
* audit

procedo con l'inserimento anche di mart, tuttavia al suo interno non saranno presenti delle tabelle, al fine di gestire al meglio gli spazi, questo schema sarà popolato in un secondo momento, ovvero quando le altre tabelle saranno funzionanti e popolate, in modo da valutare al meglio se procedere con un eventuale aumento della memoria disponibile del fornitore o non procedere con l'upgrade

In [5]:
query = """
CREATE SCHEMA IF NOT EXISTS raw;
CREATE SCHEMA IF NOT EXISTS staging;
CREATE SCHEMA IF NOT EXISTS ref;
CREATE SCHEMA IF NOT EXISTS core;
CREATE SCHEMA IF NOT EXISTS mart;
CREATE SCHEMA IF NOT EXISTS audit;
"""

execute_query(query)
print ("schemas create con sucesso")


schemas create con sucesso


Ora si procede con la creazione delle prime tabelle, sullo schema REF.


In [ ]:
query = """
CREATE TABLE IF NOT EXISTS ref.currencies (
    currency_code       CHAR(3)         PRIMARY KEY,
    currency_name       VARCHAR(50)     NOT NULL,
    currency_symbol     VARCHAR(8),
    decimal_places      SMALLINT        NOT NULL,
    is_active           BOOLEAN         NOT NULL,

    CONSTRAINT chk_currency_code_upper
        CHECK (currency_code = UPPER(currency_code)),

    CONSTRAINT chk_currency_decimal_places
        CHECK (decimal_places >= 0)
);
"""

execute_query(query)


In [23]:
# creazione della tabella Currencies
query = """
INSERT INTO ref.currencies (
    currency_code,
    currency_name,
    currency_symbol,
    decimal_places,
    is_active
)
VALUES
    ('EUR', 'Euro', '€', 2, TRUE),
    ('USD', 'US Dollar', '$', 2, TRUE),
    ('GBP', 'British Pound', '£', 2, TRUE),
    ('CHF', 'Swiss Franc', 'CHF', 2, TRUE)
ON CONFLICT (currency_code) DO NOTHING;
"""
execute_query(query)
print("Tabella popolata con successo")

Operazione completata con successo
Tabella popolata con successo


In [24]:
#Creazione della tabella Calendar
query = """
CREATE TABLE IF NOT EXISTS ref.calendar (
    date_key            INTEGER			PRIMARY KEY,
    full_date	        DATE			NOT NULL,
    year_number	        SMALLINT		NOT NULL,
    quarter_number      SMALLINT		NOT NULL,
    month_number	    SMALLINT		NOT NULL,
    month_name	        VARCHAR(15)		NOT NULL,
    week_number	        SMALLINT		NOT NULL,
    day_of_month	    SMALLINT		NOT NULL,
    day_of_week	        SMALLINT		NOT NULL,
    is_weekend	        BOOLEAN			NOT NULL,
    is_month_end	    BOOLEAN			NOT NULL,
    is_quarter_end	    BOOLEAN			NOT NULL,
    is_year_end	        BOOLEAN			NOT NULL
)
"""
execute_query(query)

Operazione completata con successo


In [9]:
#REF.countries
query = """
CREATE TABLE IF NOT EXISTS ref.countries (
    country_code        CHAR(2)	        PRIMARY KEY,
    country_name	    VARCHAR(100)	NOT NULL,
    region_name	        VARCHAR(50),
    is_active	        BOOLEAN			NOT NULL
)
"""
execute_query(query)

✓ Sweet! Data updated


In [10]:
#REF.exchanges
query = """
CREATE TABLE IF NOT EXISTS ref.exchanges (
    exchange_mic	        CHAR(4)	        PRIMARY KEY,
    exchange_name	        VARCHAR(120)	NOT NULL,
    country_code	        CHAR(2),
    timezone_name	        VARCHAR(64),
    default_currency_code	CHAR(3),
    is_active	            BOOLEAN			NOT NULL,

    CONSTRAINT fk_country FOREIGN KEY (country_code)
        REFERENCES ref.countries(country_code),
    CONSTRAINT fk_currency FOREIGN KEY (default_currency_code)
        REFERENCES ref.currencies(currency_code)
)
"""
execute_query(query)

✓ Sweet! Data updated


In [12]:
# CORE.data_sources
query = """
CREATE TABLE IF NOT EXISTS Core.data_sources (
    source_id	            BIGINT GENERATED ALWAYS AS IDENTITY	    PRIMARY KEY,
    source_code	            VARCHAR(30)			                    NOT NULL	UNIQUE,
    source_name	            VARCHAR(120)			                NOT NULL,
    source_category	        VARCHAR(30)			                    NOT NULL,
    base_url	            TEXT,
    access_method	        VARCHAR(20)			                    NOT NULL,
    authentication_type	    VARCHAR(20)			                    NOT NULL,
    license_notes	        TEXT,
    redistribution_allowed	BOOLEAN			                        NOT NULL,
    rate_limit_notes	    VARCHAR(200),
    is_official	            BOOLEAN			                        NOT NULL,
    is_active	            BOOLEAN			                        NOT NULL,
    last_verified_at	    TIMESTAMPTZ
)
"""
execute_query(query)

✓ Sweet! Data updated


In [ ]:
#da modificare per CORE.data_sources
query = """
CREATE TABLE IF NOT EXISTS Core.data_sources (
    source_id	            BIGINT GENERATED ALWAYS AS IDENTITY	    PRIMARY KEY,
    source_code	            VARCHAR(30)			                    NOT NULL	UNIQUE,
    source_name	            VARCHAR(120)			                NOT NULL,
    source_category	        VARCHAR(30)			                    NOT NULL,
    base_url	            TEXT,
    access_method	        VARCHAR(20)			                    NOT NULL,
    authentication_type	    VARCHAR(20)			                    NOT NULL,
    license_notes	        TEXT,
    redistribution_allowed	BOOLEAN			                        NOT NULL,
    rate_limit_notes	    VARCHAR(200),
    is_official	            BOOLEAN			                        NOT NULL,
    is_active	            BOOLEAN			                        NOT NULL,
    last_verified_at	    TIMESTAMPTZ
)
"""
execute_query(query)

In [17]:
#CORE.clients
query = """
CREATE TABLE IF NOT EXISTS Core.clients (
    client_id	        BIGINT GENERATED ALWAYS AS IDENTITY			PRIMARY KEY,
    client_code	        VARCHAR(30)			                        NOT NULL UNIQUE,
    client_name	        VARCHAR(150)			                    NOT NULL,
    client_type	        VARCHAR(30)			                        NOT NULL  CHECK (client_type IN ('RETAIL','PRIVATE','INSTITUTIONAL','CORPORATE', 'OTHER')),
    segment	            VARCHAR(30),
    country_code	    CHAR(2),
    risk_profile	    VARCHAR(30),
    onboarding_date	    DATE			                            NOT NULL,
    status	            VARCHAR(20)			                        NOT NULL    CHECK ( status IN ('ACTIVE','INACTIVE','CLOSED')),
    created_at	        TIMESTAMPTZ			                        NOT NULL    DEFAULT CURRENT_TIMESTAMP,
    updated_at	        TIMESTAMPTZ,

    CONSTRAINT fk_country FOREIGN KEY (country_code)
        REFERENCES ref.countries(country_code)
)
"""
execute_query(query)

✓ Sweet! Data updated


In [19]:
#CORE.issuer
query = """
CREATE TABLE IF NOT EXISTS Core.issuer (
    issuer_id	    BIGINT GENERATED ALWAYS AS IDENTITY 	PRIMARY KEY,
    issuer_code	    VARCHAR(30)			NOT NULL	                UNIQUE,
    issuer_name	    VARCHAR(150)		NOT NULL,
    issuer_type	    VARCHAR(30)			NOT NULL,
    country_code	CHAR(2),
    sector	        VARCHAR(50),
    industry	    VARCHAR(80),
    lei_code	    CHAR(20)			UNIQUE,
    is_active	    BOOLEAN			    NOT NULL,

    CONSTRAINT fk_country FOREIGN KEY (country_code)
        REFERENCES ref.countries(country_code)

)
"""
execute_query(query)

✓ Sweet! Data updated
